# Job Task 2: Validate Pipeline Output Quality

Asserts minimum row counts and acceptable data completeness on the silver and gold tables.
Raises an exception (failing the job task) if expectations are not met.

In [ ]:
dbutils.widgets.text("catalog",       "platform-workshop", "Catalog")
dbutils.widgets.text("shared_schema", "00_shared",         "Shared Schema")

catalog       = dbutils.widgets.get("catalog")
shared_schema = dbutils.widgets.get("shared_schema")

c = f"`{catalog}`"
print(f"Validating: {catalog}.{shared_schema}")

In [ ]:
checks = []

# Silver: expect at least 50 documents (we have ~130 PDFs)
silver_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_silver").collect()[0]['cnt']
checks.append(("silver row count >= 50", silver_count >= 50, silver_count))

# Silver: company field should be populated in at least 80% of rows
silver_with_company = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_silver WHERE company IS NOT NULL"
).collect()[0]['cnt']
company_pct = silver_with_company / silver_count if silver_count > 0 else 0
checks.append(("silver company completeness >= 80%", company_pct >= 0.80, f"{company_pct:.1%}"))

# Gold: expect same count as silver (1:1 mapping)
gold_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {c}.{shared_schema}.financial_docs_gold").collect()[0]['cnt']
checks.append(("gold row count == silver row count", gold_count == silver_count, gold_count))

# Summary table
print(f"\n{'Check':<45} {'Pass?':<8} {'Value'}")
print("-" * 70)
failed = []
for check, passed, value in checks:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{check:<45} {status:<8} {value}")
    if not passed:
        failed.append(check)

if failed:
    raise ValueError(f"Quality validation failed for: {failed}")
else:
    print("\n✓ All quality checks passed.")